In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#|default_exp utils

# Utils

In [ ]:
#|export
import reprlib
import warnings
from math import ceil, log10
from typing import Generator, Optional, Tuple, Union

import numpy as np
import pandas as pd

from utilsforecast.compat import DataFrame, Series, pl
from utilsforecast.data import generate_series
from utilsforecast.processing import (
    counts_by_id,
    filter_with_mask,
    group_by,
    is_in,
    offset_dates,
    take_rows,
)

In [ ]:
from fastcore.test import test_eq, test_fail
from nbdev import show_doc

In [ ]:
#|export
def generate_daily_series(
    n_series: int, 
    min_length: int = 50,
    max_length: int = 500,
    n_static_features: int = 0,
    equal_ends: bool = False,
    static_as_categorical: bool = True,
    with_trend: bool = False,
    seed: int = 0,
    engine: str = 'pandas',
) -> DataFrame:
    """Generate Synthetic Panel Series.

    Parameters
    ----------
    n_series : int
        Number of series for synthetic panel.
    min_length : int (default=50)
        Minimum length of synthetic panel's series.
    max_length : int (default=500)
        Maximum length of synthetic panel's series.
    n_static_features : int (default=0)
        Number of static exogenous variables for synthetic panel's series.
    equal_ends : bool (default=False)
        Series should end in the same date stamp `ds`.
    static_as_categorical : bool (default=True)
        Static features should have a categorical data type.        
    with_trend : bool (default=False)
        Series should have a (positive) trend.
    seed : int (default=0)
        Random seed used for generating the data.
    engine : str (default='pandas')
        Output Dataframe type.        

    Returns
    -------
    pandas or polars DataFrame
        Synthetic panel with columns [`unique_id`, `ds`, `y`] and exogenous features.
    """
    series = generate_series(
        n_series=n_series,
        freq='D',
        min_length=min_length,
        max_length=max_length,
        n_static_features=n_static_features,
        equal_ends=equal_ends,
        static_as_categorical=static_as_categorical,        
        with_trend=with_trend,
        seed=seed,
        engine=engine,
    )
    n_digits = ceil(log10(n_series))
    
    def int_id_to_str(uid):
        return f'id_{uid:0{n_digits}}'    

    if engine == 'pandas':
        series['unique_id'] = series['unique_id'].map(int_id_to_str).astype('category')
    else:
        import polars as pl

        series = series.with_columns(pl.col('unique_id').map_elements(int_id_to_str).cast(pl.Categorical))
    return series

In [ ]:
show_doc(generate_daily_series)

Generate 20 series with lengths between 100 and 1,000.

In [ ]:
n_series = 20
min_length = 100
max_length = 1000

series = generate_daily_series(n_series, min_length, max_length)
series

We can also add static features to each serie (these can be things like product_id or store_id). Only the first static feature (`static_0`) is relevant to the target.

In [ ]:
n_static_features = 2

series_with_statics = generate_daily_series(n_series, min_length, max_length, n_static_features)
series_with_statics

In [ ]:
for i in range(n_static_features):
    assert all(series_with_statics.groupby('unique_id')[f'static_{i}'].nunique() == 1)

If `equal_ends=False` (the default) then every serie has a different end date. 

In [ ]:
assert series_with_statics.groupby('unique_id')['ds'].max().nunique() > 1

We can have all of them end at the same date by specifying `equal_ends=True`.

In [ ]:
series_equal_ends = generate_daily_series(n_series, min_length, max_length, equal_ends=True)

assert series_equal_ends.groupby('unique_id')['ds'].max().nunique() == 1

In [ ]:
#|export
def generate_prices_for_series(series: pd.DataFrame, horizon: int = 7, seed: int = 0) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    unique_last_dates = series.groupby('unique_id')['ds'].max().nunique()
    if unique_last_dates > 1:
        raise ValueError('series must have equal ends.')
    day_offset = pd.tseries.frequencies.Day()
    starts_ends = series.groupby('unique_id')['ds'].agg([min, max])
    dfs = []
    for idx, (start, end) in starts_ends.iterrows():
        product_df = pd.DataFrame(
            {
                'unique_id': idx,
                'price': rng.rand((end - start).days + 1 + horizon),
            },
            index=pd.date_range(start, end + horizon * day_offset, name='ds'),
        )
        dfs.append(product_df)
    prices_catalog = pd.concat(dfs).reset_index()
    return prices_catalog

In [ ]:
series_for_prices = generate_daily_series(20, n_static_features=2, equal_ends=True)
series_for_prices.rename(columns={'static_1': 'product_id'}, inplace=True)
prices_catalog = generate_prices_for_series(series_for_prices, horizon=7)
prices_catalog

In [ ]:
test_eq(set(prices_catalog['unique_id']), set(series_for_prices['unique_id']))
test_fail(lambda: generate_prices_for_series(series), contains='equal ends')

In [ ]:
#| exporti
def single_split(
    df: DataFrame,
    i_window: int,    
    n_windows: int,
    h: int,
    id_col: str,
    time_col: str,
    freq: Union[pd.offsets.BaseOffset, int],
    max_dates: Series,  
    step_size: Optional[int] = None,
    input_size: Optional[int] = None,
) -> Tuple[DataFrame, Series, Series]:
    if step_size is None:
        step_size = h
    test_size = h + step_size * (n_windows - 1)
    offset = test_size - i_window * step_size
    train_ends = offset_dates(max_dates, freq, -offset)
    valid_ends = offset_dates(train_ends, freq, h)
    train_mask = df[time_col].le(train_ends)
    valid_mask = df[time_col].gt(train_ends) & df[time_col].le(valid_ends)    
    if input_size is not None:
        train_starts = offset_dates(train_ends, freq, -input_size)
        train_mask &= df[time_col].gt(train_starts)
    train_sizes = group_by(train_mask, df[id_col], maintain_order=True).sum()
    if isinstance(train_sizes, pd.Series):
        train_sizes = train_sizes.reset_index()
    zeros_mask = train_sizes[time_col].eq(0)   
    if zeros_mask.all():
        raise ValueError(
            'All series are too short for the cross validation settings, '
            f'at least {offset + 1} samples are required.\n'
            'Please reduce `n_windows` or `h`.'
        )
    elif zeros_mask.any():
        ids = filter_with_mask(train_sizes[id_col], zeros_mask)
        warnings.warn(
            'The following series are too short for the window '
            f'and will be dropped: {reprlib.repr(list(ids))}'
        )
        dropped_ids = is_in(df[id_col], ids)
        valid_mask &= ~dropped_ids
    last_idx_per_serie = counts_by_id(df, id_col)['counts'].to_numpy().cumsum() - 1
    cutoff_dates = take_rows(train_ends, last_idx_per_serie)
    if isinstance(cutoff_dates, pd.Series):
        cutoff_dates = cutoff_dates.reset_index()[time_col]
    cutoffs = type(df)({
        id_col: train_sizes[id_col],
        'cutoff': cutoff_dates,
    })
    return cutoffs, train_mask, valid_mask

In [ ]:
#|export
def backtest_splits(
    df: DataFrame,
    n_windows: int,
    h: int,
    id_col: str,
    time_col: str,
    freq: Union[pd.offsets.BaseOffset, int],
    step_size: Optional[int] = None,
    input_size: Optional[int] = None,
) -> Generator[Tuple[DataFrame, DataFrame, DataFrame], None, None]:
    if isinstance(df, pd.DataFrame):
        max_dates = df.groupby(id_col, observed=True)[time_col].transform('max')
    else:
        max_dates = df.select(pl.col(time_col).max().over(id_col))[time_col]
    for i in range(n_windows):
        cutoffs, train_mask, valid_mask = single_split(
            df,
            i_window=i,
            n_windows=n_windows,
            h=h,
            id_col=id_col,
            time_col=time_col,
            freq=freq,
            max_dates=max_dates,
            step_size=step_size,
            input_size=input_size,
        )
        train = filter_with_mask(df, train_mask)
        valid = filter_with_mask(df, valid_mask)
        yield cutoffs, train, valid

In [ ]:
#| hide
short_series = generate_daily_series(100, max_length=50)
backtest_results = list(
    backtest_splits(
        short_series,
        n_windows=1,
        h=49,
        id_col='unique_id',
        time_col='ds',
        freq=pd.offsets.Day(),
    )
)[0]
test_fail(
    lambda: list(
        backtest_splits(
            short_series,
            n_windows=1,
            h=50,
            id_col='unique_id',
            time_col='ds',
            freq=pd.offsets.Day(),
        )
    ),
    contains='at least 51 samples are required'
)
some_short_series = generate_daily_series(100, min_length=20, max_length=100)
with warnings.catch_warnings(record=True) as issued_warnings:
    warnings.simplefilter('always', UserWarning)
    splits = list(
        backtest_splits(
            some_short_series,
            n_windows=1,
            h=50,
            id_col='unique_id',
            time_col='ds',
            freq=pd.offsets.Day(),
        )
    )
    assert any('will be dropped' in str(w.message) for w in issued_warnings)
short_series_int = short_series.copy()
short_series_int['ds'] = short_series.groupby('unique_id').transform('cumcount')
backtest_int_results = list(
    backtest_splits(
        short_series_int,
        n_windows=1,
        h=40,
        id_col='unique_id',
        time_col='ds',
        freq=1
    )
)[0]

In [ ]:
#| hide
max_dates = series.groupby('unique_id')['ds'].max()
day_offset = pd.offsets.Day()

def test_backtest_splits(df, n_windows, h, step_size, input_size):
    common_kwargs = dict(
        n_windows=n_windows,
        h=h,
        id_col='unique_id',
        time_col='ds',
        freq=pd.offsets.Day(), 
        step_size=step_size,
        input_size=input_size,        
    )
    permuted_df = df.sample(frac=1.0)
    splits = backtest_splits(df, **common_kwargs)
    splits_on_permuted = list(backtest_splits(permuted_df, **common_kwargs))
    if step_size is None:
        step_size = h
    test_size = h + step_size * (n_windows - 1)
    for window, (cutoffs, train, valid) in enumerate(splits):
        offset = test_size - window * step_size
        expected_max_train_dates = max_dates - day_offset * offset
        max_train_dates = train.groupby('unique_id')['ds'].max()
        pd.testing.assert_series_equal(max_train_dates, expected_max_train_dates)
        pd.testing.assert_frame_equal(cutoffs, max_train_dates.rename('cutoff').reset_index())
        
        if input_size is not None:
            expected_min_train_dates = expected_max_train_dates - day_offset * (input_size - 1)
            min_train_dates = train.groupby('unique_id')['ds'].min()
            pd.testing.assert_series_equal(min_train_dates, expected_min_train_dates)

        expected_min_valid_dates = expected_max_train_dates + day_offset
        min_valid_dates = valid.groupby('unique_id')['ds'].min()
        pd.testing.assert_series_equal(min_valid_dates, expected_min_valid_dates)

        expected_max_valid_dates = expected_max_train_dates + day_offset * h
        max_valid_dates = valid.groupby('unique_id')['ds'].max()
        pd.testing.assert_series_equal(max_valid_dates, expected_max_valid_dates)

        if window == n_windows - 1:
            pd.testing.assert_series_equal(max_valid_dates, max_dates)

        _, permuted_train, permuted_valid = splits_on_permuted[window]            
        pd.testing.assert_frame_equal(train, permuted_train.sort_values(['unique_id', 'ds']))
    pd.testing.assert_frame_equal(valid, permuted_valid.sort_values(['unique_id', 'ds']))

for step_size in (None, 1, 2):
    for input_size in (None, 4):
        test_backtest_splits(series, n_windows=3, h=14, step_size=step_size, input_size=input_size)

In [ ]:
#|export
class PredictionIntervals:
    """Class for storing prediction intervals metadata information."""
    def __init__(
        self,
        n_windows: int = 2,
        h: int = 1,
        method: str = 'conformal_distribution',
    ):
        if n_windows < 2:
            raise ValueError('You need at least two windows to compute conformal intervals')
        allowed_methods = ['conformal_error', 'conformal_distribution']            
        if method not in allowed_methods:
            raise ValueError(f'method must be one of {allowed_methods}')
        self.n_windows = n_windows
        self.h = h
        self.method = method

    def __repr__(self):
        return f"PredictionIntervals(n_windows={self.n_windows}, h={self.h}, method='{self.method}')"

In [ ]:
#| exporti
def _ensure_shallow_copy(df: pd.DataFrame) -> pd.DataFrame:
    from packaging.version import Version
    
    if Version(pd.__version__) < Version("1.4"):
        # https://github.com/pandas-dev/pandas/pull/43406
        df = df.copy()
    return df

In [ ]:
#| exporti
class _ShortSeriesException(Exception):
    def __init__(self, idxs):
        self.idxs = idxs